IMPORTING LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error,log_loss,r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from scipy.stats import f_oneway
import math

In [ ]:
df = pd.read_csv('IoTpond1.csv')
df.head()

,created_at,entry_id,Temperature (C),Turbidity(NTU),Dissolved Oxygen(g/ml),PH,Ammonia(g/ml),Nitrate(g/ml),Population,Fish_Length(cm),Fish_Weight(g)
0,2021-06-19 00:00:05 CET,1889,24.8750,100,4.505,8.43365,0.45842,193,50,7.11,2.91
1,2021-06-19 00:01:02 CET,1890,24.9375,100,6.601,8.43818,0.45842,194,50,7.11,2.91
2,2021-06-19 00:01:22 CET,1891,24.8750,100,15.797,8.42457,0.45842,192,50,7.11,2.91
3,2021-06-19 00:01:44 CET,1892,24.9375,100,5.046,8.43365,0.45842,193,50,7.11,2.91
4,2021-06-19 00:02:07 CET,1893,24.9375,100,38.407,8.40641,0.45842,192,50,7.11,2.91


In [ ]:
print(df.isnull().sum())

created_at                 0
entry_id                   0
Temperature (C)            0
Turbidity(NTU)             0
Dissolved Oxygen(g/ml)     0
PH                         0
Ammonia(g/ml)             52
Nitrate(g/ml)              0
Population                 0
Fish_Length(cm)            2
Fish_Weight(g)             2
dtype: int64


In [ ]:
df.describe()

,entry_id,Temperature (C),Turbidity(NTU),Dissolved Oxygen(g/ml),PH,Ammonia(g/ml),Nitrate(g/ml),Population,Fish_Length(cm),Fish_Weight(g)
count,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,8.307400e+04,83126.000000,83126.0,83124.000000,83124.00000
mean,84018.144516,24.573376,87.490160,12.390251,7.518329,2.030817e+08,458.294408,50.0,16.414686,44.56847
std,53579.484245,0.861532,25.859375,12.518253,0.534787,7.866231e+09,338.313206,0.0,5.272244,33.21549
min,1889.000000,-127.000000,1.000000,0.007000,-0.586270,6.770000e-03,45.000000,50.0,7.110000,2.91000
25%,24902.250000,24.125000,91.000000,3.440000,7.153520,4.584200e-01,146.000000,50.0,11.790000,14.19000
50%,103478.500000,24.562500,100.000000,7.133000,7.357790,6.116600e-01,347.000000,50.0,18.080000,54.70000
75%,131074.750000,24.937500,100.000000,15.819000,7.838980,1.558803e+01,823.000000,50.0,21.000000,67.52000
max,247405.000000,27.750000,100.000000,41.046000,8.551670,4.270000e+11,1936.000000,50.0,33.450000,318.64000


In [ ]:
df['Ammonia(g/ml)'].fillna(df['Ammonia(g/ml)'].mean(), inplace=True)
df['Fish_Length(cm)'].fillna(df['Fish_Length(cm)'].mean(), inplace=True)
df['Fish_Weight(g)'].fillna(df['Fish_Weight(g)'].mean(), inplace=True)

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'], format='%Y-%m-%d %H:%M:%S %Z', utc=True)

In [ ]:
df['created_at'] = pd.to_datetime(df.created_at)

#split columns
df["Date"] = df["created_at"].dt.day
df["month"] = df["created_at"].dt.month
df['year']=df['created_at'].dt.year
df['hour']=df['created_at'].dt.hour
df['minutes']=df['created_at'].dt.minute
df['seconds']=df['created_at'].dt.second

In [ ]:
df.drop(["created_at"], axis = 1, inplace = True)

In [ ]:
 df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df['Date'] = pd.to_numeric(df['Date'])

In [ ]:
 df['month'] = pd.to_datetime(df['month'], errors='coerce')

df['month'] = pd.to_numeric(df['month'])

In [ ]:
 df['year'] = pd.to_datetime(df['year'], errors='coerce')

df['year'] = pd.to_numeric(df['year'])

In [ ]:
df['hour'] = pd.to_datetime(df['hour'], errors='coerce')

df['hour'] = pd.to_numeric(df['hour'])

In [ ]:
df['minutes'] = pd.to_datetime(df['minutes'], errors='coerce')

df['minutes'] = pd.to_numeric(df['minutes'])

In [ ]:
df['seconds'] = pd.to_datetime(df['seconds'], errors='coerce')

df['seconds'] = pd.to_numeric(df['seconds'])

In [ ]:
df.fillna(df.mean(), inplace=True)
df.replace([np.inf, -np.inf], 1e9, inplace=True)

In [ ]:
for i in range(len(df)):
    if df.loc[i, "Temperature (C)"] < 20 or df.loc[i, "Temperature (C)"] > 30:
        df.loc[i, "Temperature (C)"] = df.loc[i-1, "Temperature (C)"]

In [ ]:
for i in range(1,len(df)):
    if df.loc[i, "PH"] < 6 or df.loc[i, "PH"] > 8.5:
        df.loc[i, "PH"] = df.loc[i-1,'PH']

In [ ]:
df.at[0, 'Dissolved Oxygen(g/ml)'] = 5

In [ ]:
for i in range(1,len(df)):
    if df.loc[i, "Dissolved Oxygen(g/ml)"] < 5 or df.loc[i, "Dissolved Oxygen(g/ml)"] > 20:
        df.loc[i, "Dissolved Oxygen(g/ml)"] = df.loc[i-1, "Dissolved Oxygen(g/ml)"]

In [ ]:
df.describe()

,entry_id,Temperature (C),Turbidity(NTU),Dissolved Oxygen(g/ml),PH,Ammonia(g/ml),Nitrate(g/ml),Population,Fish_Length(cm),Fish_Weight(g),Date,month,year,hour,minutes,seconds
count,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,8.312600e+04,83126.000000,83126.0,83126.000000,83126.000000,83126.000000,83126.000000,83126.0,83126.000000,83126.000000,83126.000000
mean,84018.144516,24.575198,87.490160,10.610544,7.533544,2.030817e+08,458.294408,50.0,16.414686,44.568470,15.911123,7.158964,2021.0,11.580288,29.529870,29.530568
std,53579.484245,0.682531,25.859375,4.701762,0.445071,7.863770e+09,338.313206,0.0,5.272180,33.215091,9.535208,0.815633,0.0,6.838421,17.325825,17.319509
min,1889.000000,23.000000,1.000000,5.000000,6.068580,6.770000e-03,45.000000,50.0,7.110000,2.910000,1.000000,6.000000,2021.0,0.000000,0.000000,0.000000
25%,24902.250000,24.125000,91.000000,6.588000,7.153520,4.584200e-01,146.000000,50.0,11.790000,14.190000,9.000000,6.000000,2021.0,6.000000,15.000000,15.000000
50%,103478.500000,24.562500,100.000000,9.078000,7.357790,6.139700e-01,347.000000,50.0,18.080000,54.700000,14.000000,7.000000,2021.0,12.000000,30.000000,29.000000
75%,131074.750000,24.937500,100.000000,14.368000,7.838980,1.565161e+01,823.000000,50.0,21.000000,67.520000,24.000000,8.000000,2021.0,17.000000,45.000000,45.000000
max,247405.000000,27.750000,100.000000,19.992000,8.492660,4.270000e+11,1936.000000,50.0,33.450000,318.640000,31.000000,10.000000,2021.0,23.000000,59.000000,59.000000


In [ ]:
df["Ammonia(g/ml)"] = np.log(df["Ammonia(g/ml)"]) / 10.0
df["Ammonia(g/ml)"] = df["Ammonia(g/ml)"].abs()


In [ ]:
df["Nitrate(g/ml)"] = df["Nitrate(g/ml)"].apply(np.sqrt)

In [ ]:
df.describe()

,entry_id,Temperature (C),Turbidity(NTU),Dissolved Oxygen(g/ml),PH,Ammonia(g/ml),Nitrate(g/ml),Population,Fish_Length(cm),Fish_Weight(g),Date,month,year,hour,minutes,seconds
count,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.0,83126.000000,83126.000000,83126.000000,83126.000000,83126.0,83126.000000,83126.000000,83126.000000
mean,84018.144516,24.575198,87.490160,10.610544,7.533544,0.270844,19.880695,50.0,16.414686,44.568470,15.911123,7.158964,2021.0,11.580288,29.529870,29.530568
std,53579.484245,0.682531,25.859375,4.701762,0.445071,0.385938,7.940599,0.0,5.272180,33.215091,9.535208,0.815633,0.0,6.838421,17.325825,17.319509
min,1889.000000,23.000000,1.000000,5.000000,6.068580,0.000036,6.708204,50.0,7.110000,2.910000,1.000000,6.000000,2021.0,0.000000,0.000000,0.000000
25%,24902.250000,24.125000,91.000000,6.588000,7.153520,0.077997,12.083046,50.0,11.790000,14.190000,9.000000,6.000000,2021.0,6.000000,15.000000,15.000000
50%,103478.500000,24.562500,100.000000,9.078000,7.357790,0.082137,18.627936,50.0,18.080000,54.700000,14.000000,7.000000,2021.0,12.000000,30.000000,29.000000
75%,131074.750000,24.937500,100.000000,14.368000,7.838980,0.275465,28.687977,50.0,21.000000,67.520000,24.000000,8.000000,2021.0,17.000000,45.000000,45.000000
max,247405.000000,27.750000,100.000000,19.992000,8.492660,2.678005,44.000000,50.0,33.450000,318.640000,31.000000,10.000000,2021.0,23.000000,59.000000,59.000000


In [ ]:
columns=df.columns.tolist()
columns=columns[-6:]+columns[0:-6]
df=df[columns]

In [ ]:
df.head()

,Date,month,year,hour,minutes,seconds,entry_id,Temperature (C),Turbidity(NTU),Dissolved Oxygen(g/ml),PH,Ammonia(g/ml),Nitrate(g/ml),Population,Fish_Length(cm),Fish_Weight(g)
0,18,6,2021,22,0,5,1889,24.8750,100,5.000,8.43365,0.077997,13.892444,50,7.11,2.91
1,18,6,2021,22,1,2,1890,24.9375,100,6.601,8.43818,0.077997,13.928388,50,7.11,2.91
2,18,6,2021,22,1,22,1891,24.8750,100,15.797,8.42457,0.077997,13.856406,50,7.11,2.91
3,18,6,2021,22,1,44,1892,24.9375,100,5.046,8.43365,0.077997,13.892444,50,7.11,2.91
4,18,6,2021,22,2,7,1893,24.9375,100,5.046,8.40641,0.077997,13.856406,50,7.11,2.91


In [ ]:
df.describe()

,Date,month,year,hour,minutes,seconds,entry_id,Temperature (C),Turbidity(NTU),Dissolved Oxygen(g/ml),PH,Ammonia(g/ml),Nitrate(g/ml),Population,Fish_Length(cm),Fish_Weight(g)
count,83126.000000,83126.000000,83126.0,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.000000,83126.0,83126.000000,83126.000000
mean,15.911123,7.158964,2021.0,11.580288,29.529870,29.530568,84018.144516,24.575198,87.490160,10.610544,7.533544,0.270844,19.880695,50.0,16.414686,44.568470
std,9.535208,0.815633,0.0,6.838421,17.325825,17.319509,53579.484245,0.682531,25.859375,4.701762,0.445071,0.385938,7.940599,0.0,5.272180,33.215091
min,1.000000,6.000000,2021.0,0.000000,0.000000,0.000000,1889.000000,23.000000,1.000000,5.000000,6.068580,0.000036,6.708204,50.0,7.110000,2.910000
25%,9.000000,6.000000,2021.0,6.000000,15.000000,15.000000,24902.250000,24.125000,91.000000,6.588000,7.153520,0.077997,12.083046,50.0,11.790000,14.190000
50%,14.000000,7.000000,2021.0,12.000000,30.000000,29.000000,103478.500000,24.562500,100.000000,9.078000,7.357790,0.082137,18.627936,50.0,18.080000,54.700000
75%,24.000000,8.000000,2021.0,17.000000,45.000000,45.000000,131074.750000,24.937500,100.000000,14.368000,7.838980,0.275465,28.687977,50.0,21.000000,67.520000
max,31.000000,10.000000,2021.0,23.000000,59.000000,59.000000,247405.000000,27.750000,100.000000,19.992000,8.492660,2.678005,44.000000,50.0,33.450000,318.640000


In [ ]:
df.isna().sum()

Date                      0
month                     0
year                      0
hour                      0
minutes                   0
seconds                   0
entry_id                  0
Temperature (C)           0
Turbidity(NTU)            0
Dissolved Oxygen(g/ml)    0
PH                        0
Ammonia(g/ml)             0
Nitrate(g/ml)             0
Population                0
Fish_Length(cm)           0
Fish_Weight(g)            0
dtype: int64

In [ ]:
df.drop(columns=['entry_id','Population'],axis=1,inplace=True)

In [ ]:
df.to_csv('pp1.csv', index=False)

In [ ]:
df['Dissolved Oxygen(g/ml)'].min()

5.0